In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

: 

In [ ]:
from langchain_groq import ChatGroq


In [8]:
llmGroq=ChatGroq(
  model="llama-3.3-70b-versatile",
  api_key=os.getenv("GROQ_API_KEY"),
  temperature=0.2,
)


# Ouput parser

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

json_prompt=PromptTemplate.from_template(
    "You are a helpful assistant. Please provide a JSON response with the following structure: {format_instructions}"
)

json_parser=SimpleJsonOutputParser()
json_chain=json_prompt | llmGroq | json_parser

In [14]:
res=json_chain.invoke({
  "format_instructions":"What is biggest country?"
})

In [15]:
res

{'question': 'What is biggest country?',
 'answer': 'Russia',
 'details': {'country': 'Russia',
  'land_area': '17,125,200 square kilometers',
  'percentage_of_world_land_area': '11%'}}

## Basic RAG implementation with LCEL

In [14]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load environment variables once
load_dotenv()

# 1. Load Document Data Loader
docs = PyPDFLoader("LangChain.pdf").load()

# 2. Split Document
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
  
# 3. Embed + Store (In-Memory)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings)

# 4. Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 5. Initialize LLM (API key is pulled automatically from .env)
llmGroq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
)

prompt=ChatPromptTemplate.from_template(
    "Answer using only this context:\n{context}\n\nQuestion: {question}"
)

# 6. Query and Retrieve
query = "Summarize the key point of my notes"
relevant_chunks = retriever.invoke(query)
context = "\n\n".join(d.page_content for d in relevant_chunks)
print("Context:\n", context)

# 7. Construct Prompt and Chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question using ONLY the provided context. If you do not know the answer based on the context, say 'I cannot find that in the notes.'"),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
])
query="Summarize the key point of my notes"
chain = prompt | llmGroq | StrOutputParser()
response = chain.invoke({"context": context, "question": query})

print(response)


C:\Users\impav\AppData\Local\Temp\ipykernel_26012\423549346.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2274.28it/s]


Context:
 4 Vasilios Mavroudis
Memory: Enables applications to retain information from past interactions,
supporting both basic and advanced memory structures. This component is crit-
ical for maintaining context across sessions and delivering contextually aware
responses.
Indexes: Serve as structured databases that organize and store information,
allowing for efficient data retrieval when processing language queries.
Retrievers: Designed to work alongside indexes, retrievers fetch relevant data

(LLM), which then generates an accurate and contextually enriched answer. This ar-
chitecture enhances the model’s ability to produce factually grounded responses by
incorporating relevant knowledge from the vector store.
The rest of this section provides an overview of LangChain’s primary com-
ponents, followed by a brief introduction to its advanced modules–LangSmith,
LangGraph and LangServe–which are further discussed in Sections 2, 3, and 4
respectively:

vice agents or educational tools t

# Temporary Memory (in-memory, session-only)

### RunnableWithMessageHistory is a LangChain wrapper class that automatically manages and injects conversation history into a chain.

In [6]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Simple in-memory store per session
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    llmGroq,
    get_session_history,
)

response = chain_with_memory.invoke(
    [{"role": "user", "content": "My name is Rahul"}],
    config={"configurable": {"session_id": "user123"}}
)

response2 = chain_with_memory.invoke(
    [{"role": "user", "content": "What's my name?"}],
    config={"configurable": {"session_id": "user123"}}
)
print(response)
print(response2.content) 

d:\Ai Engineer_\Week1\day6\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


content="Hello Rahul! It's nice to meet you. Is there something I can help you with or would you like to chat?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 39, 'total_tokens': 65, 'completion_time': 0.04359626, 'completion_tokens_details': None, 'prompt_time': 0.00347817, 'prompt_tokens_details': None, 'queue_time': 0.057307965, 'total_time': 0.04707443}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fa4e4-4b8e-76b3-8714-e5d4944dec55-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 26, 'total_tokens': 65}
Your name is Rahul.


# Permanent Memory

In [ ]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_history.db"
    )

chain_with_memory = RunnableWithMessageHistory(
    llmGroq,
    get_session_history,
)

### Mini Project - Smart Query Router

In [4]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnablePassthrough
load_dotenv()


True

In [6]:
llmGroq=ChatGroq(
  model="llama-3.3-70b-versatile",
  api_key=os.getenv("GROQ_API_KEY"),
  temperature=0.2,
)


In [25]:
# 1. Math chain - locked to concise numeric answers
math_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math tutor. Give ONLY the final numeric answer and a one-line explanation. No fluff."),
    ("user", "{question}")
])
math_chain = math_prompt | llmGroq.bind(temperature=0) | StrOutputParser()
# .bind(temperature=0) forces deterministic math answers regardless of the LLM's default temperature

# 2. Code chain - locked to code-focused responses
code_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior software engineer. Answer with a code example first, then a brief explanation."),
    ("user", "{question}")
])
code_chain = code_prompt | llmGroq | StrOutputParser()

# 3. General chain - plain conversational assistant
general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly, helpful assistant. Answer clearly and concisely."),
    ("user", "{question}")
])
def log_general_route(input_dict: dict) -> dict:
    """Logs fallback routing before passing payload forward."""
    print("[ROUTER LOG]: 💬 Routed to GENERAL CHAIN (Default Fallback)",flush=True)
    return input_dict

general_chain = (
    RunnableLambda(log_general_route) 
    | general_prompt 
    | llmGroq 
    | StrOutputParser()
)

In [26]:
def is_math_question(input_dict):
    question = input_dict["question"].lower()
    keywords = ["calculate", "solve", "how much is", "+", "-", "*", "/", "sum", "average"]
    return any(k in question for k in keywords)

def is_code_question(input_dict):
    question = input_dict["question"].lower()
    keywords = ["code", "python", "function", "bug", "error", "script", "debug"]
    return any(k in question for k in keywords)

router_chain = RunnableBranch(
    (is_math_question, math_chain),
    (is_code_question, code_chain),
    general_chain  # default fallback if neither condition matches
)

In [28]:
questions = [
    "   Calculate 245 * 12   ",
    "Write a Python function to reverse a string",
    "What's the capital of France?",
    "Hi, How are you doing today?"
]

for q in questions:
    print(f"\n--- Question: {q} ---")
    result = router_chain.invoke({"question": q})
    print(result)


--- Question:    Calculate 245 * 12    ---
2940, result of multiplying 245 by 12.

--- Question: Write a Python function to reverse a string ---
```python
def reverse_string(input_str: str) -> str:
    """
    Reverses the input string.

    Args:
        input_str (str): The string to be reversed.

    Returns:
        str: The reversed string.
    """
    return input_str[::-1]

# Example usage:
print(reverse_string("Hello World"))  # Output: "dlroW olleH"
```

This function uses Python's slice notation to extract the characters of the input string in reverse order. The `[::-1]` slice means "start at the end of the string and end at position 0, move with the step -1" which effectively reverses the string.

--- Question: What's the capital of France? ---
[ROUTER LOG]: 💬 Routed to GENERAL CHAIN (Default Fallback)
The capital of France is Paris.

--- Question: Hi, How are you doing today? ---
[ROUTER LOG]: 💬 Routed to GENERAL CHAIN (Default Fallback)
I'm doing well, thanks for asking. 

### Chain with just RunnablePassthrough() will output the original input without any modification.

In [30]:
from langchain_core.runnables import RunnablePassthrough
chain = RunnablePassthrough()

In [31]:
chain.invoke("Hello")

'Hello'

### RunnableLambda

 -To use a custom function inside a LCEL chain we need to wrap it up with RunnableLambda.


In [32]:
def greeting(name: str) -> str:
    return f"{name} is a great name!"

In [33]:
from langchain_core.runnables import RunnableLambda

chain=RunnablePassthrough() | RunnableLambda(greeting)

In [34]:
chain.invoke("Harsh")

'Harsh is a great name!'

## RunnableParallel

We will use `RunnableParallel` for running tasks in parallel. This is one of the most important and useful `Runnable` classes in LangChain.

In the following chain, `RunnableParallel` executes two tasks concurrently:
* **`operation_a`**: Uses `RunnablePassthrough` to forward the raw input.
* **`operation_b`**: Uses `RunnableLambda` wrapped around the `russian_lastname` function.

In [36]:
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "operation_b": RunnableLambda(greeting)
    }
)

In [37]:
chain.invoke("Harsh")

{'operation_a': 'Harsh', 'operation_b': 'Harsh is a great name!'}

In [39]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq  
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# Free, local embeddings — no API key needed
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_texts(
    ["MLCourses focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."],
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llmGroq
    | StrOutputParser()
)

response = retrieval_chain.invoke("What is MLCourses?")
print(response)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3087.71it/s]


MLCourses focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English.


### Combining LCEL Chains

: 